# SPA vs Baselines — Results Analysis
Run this notebook from `~/FedLLM-Re/rework/` on the server.

In [ ]:
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

# V1 = old runs (homo_r4, homo_r8, hetero_pad, flexlora, hetero_spa, spa_m)
# V2 = new runs (all methods re-run + hetlora)
RESULTS_DIRS = [
    "../results/yelp",
    "../results/V2",
    "../results_v2/yelp",
]

METHODS = ["homo_r4", "homo_r8", "hetero_pad", "flexlora", "hetero_spa", "hetlora", "spa_m"]
LABELS  = {
    "homo_r4":    "FedAvg r=4",
    "homo_r8":    "FedAvg r=8",
    "hetero_pad": "Hetero-Pad",
    "flexlora":   "FlexLoRA",
    "hetero_spa": "SPA",
    "hetlora":    "HetLoRA",
    "spa_m":      "SPA-M (Ours)",
}
COLORS = {
    "homo_r4":    "#9e9e9e",
    "homo_r8":    "#607d8b",
    "hetero_pad": "#ff7043",
    "flexlora":   "#42a5f5",
    "hetero_spa": "#e53935",
    "hetlora":    "#ff7f0e",
    "spa_m":      "#6a0dad",
}
LS = {
    "homo_r4": ":", "homo_r8": "--",
    "hetero_pad": "-.", "flexlora": "--",
    "hetero_spa": "-.", "hetlora": (0,(3,1,1,1)), "spa_m": "-"
}
LW = {
    "homo_r4": 1.2, "homo_r8": 1.2,
    "hetero_pad": 1.5, "flexlora": 2.0,
    "hetero_spa": 1.8, "hetlora": 2.0, "spa_m": 2.5
}

def load_all():
    data = defaultdict(lambda: defaultdict(list))
    seen = set()  # deduplicate by (method, alpha, seed)
    for results_dir in RESULTS_DIRS:
        if not os.path.isdir(results_dir):
            continue
        files = glob.glob(os.path.join(results_dir, "*.json"))
        files = [f for f in files if "(1)" not in f]
        for f in files:
            with open(f) as fh:
                d = json.load(fh)
            method = d.get("method", "")
            alpha  = d.get("alpha", None)
            seed   = d.get("seed", None)
            if method not in METHODS:
                continue
            key = (method, alpha, seed)
            if key in seen:
                continue
            seen.add(key)
            data[(method, alpha)]["runs"].append(d)
    return data

def get_curve(runs, metric, scale=1.0):
    per_round = defaultdict(list)
    for run in runs:
        for r in run["rounds"]:
            if metric in r:
                per_round[r["round"]].append(r[metric] * scale)
    rounds, means, stds = [], [], []
    for rnd in sorted(per_round):
        vals = per_round[rnd]
        rounds.append(rnd)
        means.append(np.mean(vals))
        stds.append(np.std(vals))
    return np.array(rounds), np.array(means), np.array(stds)

def summary_metrics(runs, metrics):
    """Compute Mean-L5, Final, and Best for each metric across seeds."""
    result = {}
    for metric in metrics:
        ml5s, fins, bsts = [], [], []
        for run in runs:
            vals = [r.get(metric, np.nan) for r in run["rounds"]]
            vals = [v for v in vals if not np.isnan(v)]
            if not vals:
                continue
            ml5s.append(np.mean(vals[-5:]))
            fins.append(vals[-1])
            bsts.append(max(vals))
        if ml5s:
            result[metric] = {
                "mean_l5": (np.mean(ml5s), np.std(ml5s), len(ml5s)),
                "final":   (np.mean(fins), np.std(fins), len(fins)),
                "best":    (np.mean(bsts), np.std(bsts), len(bsts)),
            }
        else:
            nan3 = (np.nan, np.nan, 0)
            result[metric] = {"mean_l5": nan3, "final": nan3, "best": nan3}
    return result

DATA = load_all()
available = sorted(DATA.keys())
print("Available results:")
for (m, a) in available:
    n = len(DATA[(m,a)]["runs"])
    seeds = sorted(r["seed"] for r in DATA[(m,a)]["runs"])
    print(f"  {m:15s}  alpha={a}  n={n}  seeds={seeds}")

## 1. Convergence Curves — Accuracy

In [ ]:
alphas = sorted(set(a for _, a in available))
fig, axes = plt.subplots(1, len(alphas), figsize=(7*len(alphas), 5), sharey=False)
if len(alphas) == 1:
    axes = [axes]

for ax, alpha in zip(axes, alphas):
    for method in METHODS:
        key = (method, alpha)
        if key not in DATA:
            continue
        runs = DATA[key]["runs"]
        rounds, means, stds = get_curve(runs, "accuracy", scale=100)
        if len(rounds) == 0:
            continue
        ax.plot(rounds, means,
                label=LABELS[method], color=COLORS[method],
                linestyle=LS[method], linewidth=LW[method])
        if len(runs) > 1:
            ax.fill_between(rounds, means-stds, means+stds,
                            alpha=0.12, color=COLORS[method])
    ax.set_title(f"Yelp Accuracy — α={alpha}", fontsize=13)
    ax.set_xlabel("Round", fontsize=11)
    ax.set_ylabel("Accuracy (%)", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/convergence_accuracy.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/convergence_accuracy.pdf")

## 2. Convergence Curves — F1-macro

In [ ]:
fig, axes = plt.subplots(1, len(alphas), figsize=(7*len(alphas), 5), sharey=False)
if len(alphas) == 1:
    axes = [axes]

for ax, alpha in zip(axes, alphas):
    for method in METHODS:
        key = (method, alpha)
        if key not in DATA:
            continue
        runs = DATA[key]["runs"]
        rounds, means, stds = get_curve(runs, "f1_macro", scale=100)
        if len(rounds) == 0:
            continue
        ax.plot(rounds, means,
                label=LABELS[method], color=COLORS[method],
                linestyle=LS[method], linewidth=LW[method])
        if len(runs) > 1:
            ax.fill_between(rounds, means-stds, means+stds,
                            alpha=0.12, color=COLORS[method])
    ax.set_title(f"Yelp F1-macro — α={alpha}", fontsize=13)
    ax.set_xlabel("Round", fontsize=11)
    ax.set_ylabel("F1-macro (%)", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("figures/convergence_f1.pdf", bbox_inches="tight", dpi=150)
plt.show()

## 3. Perplexity Convergence

In [ ]:
fig, axes = plt.subplots(1, len(alphas), figsize=(7*len(alphas), 5), sharey=False)
if len(alphas) == 1:
    axes = [axes]

for ax, alpha in zip(axes, alphas):
    for method in METHODS:
        key = (method, alpha)
        if key not in DATA:
            continue
        runs = DATA[key]["runs"]
        rounds, means, stds = get_curve(runs, "perplexity")
        if len(rounds) == 0:
            continue
        ax.plot(rounds, means,
                label=LABELS[method], color=COLORS[method],
                linestyle=LS[method], linewidth=LW[method])
        if len(runs) > 1:
            ax.fill_between(rounds, means-stds, means+stds,
                            alpha=0.12, color=COLORS[method])
    ax.set_title(f"Yelp Perplexity — α={alpha}", fontsize=13)
    ax.set_xlabel("Round", fontsize=11)
    ax.set_ylabel("Perplexity (lower=better)", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("figures/convergence_perplexity.pdf", bbox_inches="tight", dpi=150)
plt.show()

## 4. SPA vs FlexLoRA Head-to-Head

In [ ]:
fig, axes = plt.subplots(1, len(alphas), figsize=(7*len(alphas), 5))
if len(alphas) == 1:
    axes = [axes]

focus = ["flexlora", "hetero_spa", "hetlora", "spa_m"]

for ax, alpha in zip(axes, alphas):
    for method in focus:
        key = (method, alpha)
        if key not in DATA:
            continue
        runs = DATA[key]["runs"]
        rounds, means, stds = get_curve(runs, "accuracy", scale=100)
        ax.plot(rounds, means,
                label=f"{LABELS[method]} (n={len(runs)})",
                color=COLORS[method],
                linestyle=LS[method], linewidth=LW[method])
        if len(runs) > 1:
            ax.fill_between(rounds, means-stds, means+stds,
                            alpha=0.15, color=COLORS[method])
    ax.set_title(f"SPA-M vs SPA vs FlexLoRA vs HetLoRA — α={alpha}", fontsize=12)
    ax.set_xlabel("Round", fontsize=11)
    ax.set_ylabel("Accuracy (%)", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("figures/spa_vs_baselines.pdf", bbox_inches="tight", dpi=150)
plt.show()

## 5. Final Round Summary Table

In [ ]:
METRICS = ["accuracy", "f1_macro", "perplexity"]
SCALE   = {"accuracy": 100, "f1_macro": 100, "perplexity": 1}
HIGHER  = {"accuracy", "f1_macro"}

alphas = sorted(set(a for _, a in DATA.keys()))

for alpha in alphas:
    print(f"\n{'='*82}")
    print(f"YELP — α={alpha}  |  ★=Mean-L5  (primary metric)")
    print(f"{'='*82}")
    print(f"{'Method':<18} {'Mean-L5★':>12} {'Final':>12} {'Best':>12}  {'n':>3}")
    print("-"*58)

    rows = {}
    for method in METHODS:
        key = (method, alpha)
        if key not in DATA:
            continue
        rows[method] = summary_metrics(DATA[key]["runs"], METRICS)

    # Find best Mean-L5 accuracy per column
    best_ml5 = {}
    for metric in METRICS:
        vals = [(m, rows[m][metric]["mean_l5"][0]) for m in rows
                if not np.isnan(rows[m][metric]["mean_l5"][0])]
        if vals:
            best_ml5[metric] = (max if metric in HIGHER else min)(vals, key=lambda x: x[1])[0]

    for method in METHODS:
        if method not in rows:
            continue
        sm = rows[method]["accuracy"]
        n  = len(DATA[(method, alpha)]["runs"])

        def fmt(tup, s=100):
            m, std, _ = tup
            return f"{'—':>8}" if np.isnan(m) else f"{m*s:>6.1f}±{std*s:.1f}"

        star = "★" if best_ml5.get("accuracy") == method else " "
        marker = " <<<" if method == "spa_m" else ("  *" if method == "hetlora" else "")
        print(f"  {LABELS[method]:<16} {fmt(sm['mean_l5'])}{star}  {fmt(sm['final'])}   {fmt(sm['best'])}   {n}{marker}")

print("\n★ = best Mean-L5  |  <<< = our method  |  * = new baseline")

## 6. Bar Chart — Final Accuracy Comparison

In [ ]:
alphas = sorted(set(a for _, a in DATA.keys()))
methods_avail = [m for m in METHODS if any((m, a) in DATA for a in alphas)]
x = np.arange(len(methods_avail))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

for i, alpha in enumerate(alphas):
    means, errs = [], []
    for method in methods_avail:
        key = (method, alpha)
        if key in DATA:
            sm = summary_metrics(DATA[key]["runs"], ["accuracy"])["accuracy"]
            m, s, _ = sm["mean_l5"]
            means.append(m * 100 if not np.isnan(m) else 0)
            errs.append(s * 100 if not np.isnan(s) else 0)
        else:
            means.append(0); errs.append(0)
    offset = (i - len(alphas)/2 + 0.5) * width
    ax.bar(x + offset, means, width,
           label=f"α={alpha}",
           yerr=errs, capsize=4, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([LABELS[m] for m in methods_avail], fontsize=9, rotation=15, ha="right")
ax.set_ylabel("Mean-L5 Accuracy ★ (%)", fontsize=12)
ax.set_title("Yelp — Mean-L5 Accuracy by Method (last 5 rounds avg)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top","right"]].set_visible(False)
ax.set_ylim(0, 70)

plt.tight_layout()
plt.savefig("figures/bar_meanl5_accuracy.pdf", bbox_inches="tight", dpi=150)
plt.show()

## 7. Per-Seed Scatter — SPA vs FlexLoRA

In [ ]:
fig, axes = plt.subplots(1, len(alphas), figsize=(6*len(alphas), 5))
if len(alphas) == 1:
    axes = [axes]

for ax, alpha in zip(axes, alphas):
    spa_key  = ("hetero_spa", alpha)
    flex_key = ("flexlora",   alpha)
    if spa_key not in DATA or flex_key not in DATA:
        ax.set_title(f"No data for α={alpha}")
        continue

    spa_runs  = {r["seed"]: r["rounds"][-1]["accuracy"]*100 for r in DATA[spa_key]["runs"]}
    flex_runs = {r["seed"]: r["rounds"][-1]["accuracy"]*100 for r in DATA[flex_key]["runs"]}
    seeds = sorted(set(spa_runs) & set(flex_runs))

    spa_vals  = [spa_runs[s]  for s in seeds]
    flex_vals = [flex_runs[s] for s in seeds]

    # Scatter
    ax.scatter(flex_vals, spa_vals, s=100, zorder=5,
               color="#e53935", edgecolors="black", linewidth=0.8)
    for i, s in enumerate(seeds):
        ax.annotate(f"s{s}", (flex_vals[i], spa_vals[i]),
                    textcoords="offset points", xytext=(5,3), fontsize=9)

    # Diagonal = equal performance
    lims = [min(flex_vals+spa_vals)-2, max(flex_vals+spa_vals)+2]
    ax.plot(lims, lims, "k--", alpha=0.4, linewidth=1, label="Equal")
    ax.fill_between(lims, lims, [lims[1]]*2, alpha=0.05, color="red", label="SPA better")
    ax.fill_between(lims, [lims[0]]*2, lims, alpha=0.05, color="blue", label="FlexLoRA better")

    ax.set_xlabel("FlexLoRA Final Accuracy (%)", fontsize=11)
    ax.set_ylabel("SPA Final Accuracy (%)", fontsize=11)
    ax.set_title(f"Per-seed comparison — α={alpha}", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("figures/scatter_spa_vs_flexlora.pdf", bbox_inches="tight", dpi=150)
plt.show()

## 8. LaTeX Table (paste directly into paper)

In [ ]:
print("% ============================================================")
print("% Table: Yelp Results — Mean-L5★ / Best Acc (primary metrics)")
print("% ============================================================")
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\begin{tabular}{lcccc}")
print(r"\toprule")
print(r" & \multicolumn{2}{c}{$\alpha=0.5$} & \multicolumn{2}{c}{$\alpha=0.1$} \\")
print(r"\cmidrule(lr){2-3} \cmidrule(lr){4-5}")
print(r"Method & Mean-L5$\uparrow$ & Best Acc$\uparrow$ & Mean-L5$\uparrow$ & Best Acc$\uparrow$ \\")
print(r"\midrule")

for method in METHODS:
    cells = []
    for alpha in [0.5, 0.1]:
        key = (method, alpha)
        if key in DATA:
            sm = summary_metrics(DATA[key]["runs"], ["accuracy"])["accuracy"]
            ml5_m, ml5_s, n = sm["mean_l5"]
            bst_m, bst_s, _ = sm["best"]
            if np.isnan(ml5_m):
                cells += ["—", "—"]
            else:
                cells.append(f"{ml5_m*100:.1f}$_{{\\pm {ml5_s*100:.1f}}}$ (n={n})")
                cells.append(f"{bst_m*100:.1f}$_{{\\pm {bst_s*100:.1f}}}$")
        else:
            cells += ["—", "—"]

    label = LABELS[method]
    if method == "spa_m":
        label = r"\textbf{" + label + r"}"
    elif method == "hetlora":
        label = r"\textit{" + label + r"}"
    row = f"{label} & " + " & ".join(cells) + r" \\"
    if method in ("hetero_spa", "spa_m"):
        row = r"\rowcolor{gray!10} " + row
    print(row)

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\caption{Yelp-5 results. Mean-L5★ = mean accuracy over last 5 rounds (primary). Best Acc = peak across all rounds. Bold = ours. Italic = new baseline (HetLoRA).}")
print(r"\label{tab:yelp}")
print(r"\end{table}")

## 9. Best-Round Accuracy — Same 3 Seeds (43, 44, 45)

In [ ]:
FIXED_SEEDS = [43, 44, 45]
METRICS = ["accuracy", "f1_macro", "perplexity"]
SCALE   = {"accuracy": 100, "f1_macro": 100, "perplexity": 1}
HIGHER  = {"accuracy", "f1_macro"}

def best_round_metrics(runs, metrics, seeds):
    """For each run matching a seed, take the best round value instead of final round."""
    filtered = [r for r in runs if r["seed"] in seeds]
    result = {}
    for metric in metrics:
        vals = []
        for run in filtered:
            round_vals = [r[metric] for r in run["rounds"] if metric in r]
            if round_vals:
                best = max(round_vals) if metric in HIGHER else min(round_vals)
                vals.append(best)
        if vals:
            result[metric] = (np.mean(vals), np.std(vals), len(vals))
        else:
            result[metric] = (np.nan, np.nan, 0)
    return result

alphas = sorted(set(a for _, a in DATA.keys()))

for alpha in alphas:
    print(f"\n{'='*72}")
    print(f"YELP — α={alpha} — BEST Round (seeds {FIXED_SEEDS}, n=3 each)")
    print(f"{'='*72}")
    print(f"{'Method':<18} {'Best Acc':>14} {'Best F1':>14} {'Best PPL':>14}")
    print("-"*72)

    rows = {}
    for method in METHODS:
        key = (method, alpha)
        if key not in DATA:
            continue
        runs = DATA[key]["runs"]
        available_seeds = [r["seed"] for r in runs]
        usable = [s for s in FIXED_SEEDS if s in available_seeds]
        if len(usable) == 0:
            continue
        rows[method] = best_round_metrics(runs, METRICS, usable)

    best = {}
    for metric in METRICS:
        vals = [(m, rows[m][metric][0]) for m in rows if not np.isnan(rows[m][metric][0])]
        if vals:
            best[metric] = max(vals, key=lambda x: x[1])[0] if metric in HIGHER else min(vals, key=lambda x: x[1])[0]

    for method in METHODS:
        if method not in rows:
            continue
        fm = rows[method]
        cells = []
        for metric in METRICS:
            mean, std, n = fm[metric]
            s = SCALE[metric]
            if np.isnan(mean):
                cells.append(f"{'—':>14}")
            else:
                tag = "*" if best.get(metric) == method else " "
                cells.append(f"{mean*s:>6.2f}±{std*s:<5.2f}{tag}")
        marker = " <<<" if method == "spa_m" else ""
        print(f"{LABELS[method]:<18} {'  '.join(cells)}{marker}")

print("\n* = best in column")